# 02 — CNN from Scratch (Experiment A0)

**ICS555 Capstone: African Landmark Recognition**

Trains a 3-block VGG-inspired CNN from scratch as the baseline.
This corresponds to experiment **A0** in the ablation matrix.


**Run:** `python ablation_runner.py --config configs/A0_scratch.yaml`

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/ajegetina/african-landmark-recognition.git
    %cd african-landmark-recognition
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount('/content/drive')
    !ln -sf /content/drive/MyDrive/landmark_images landmark_images

    import wandb
    wandb.login()

In [ ]:
%matplotlib inline
import sys
import torch
sys.path.insert(0, '..')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — running on CPU')

## 1. Data

In [ ]:
from src.data import get_data_loaders, visualize_one_batch

data_loaders = get_data_loaders(batch_size=64, valid_size=0.2, num_workers=2)
visualize_one_batch(data_loaders)

## 2. Model architecture

In [ ]:
from src.model import MyModel

model = MyModel(num_classes=50)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(model)

## 3. Loss and optimiser

In [ ]:
from src.optimization import get_loss, get_optimizer

loss_fn   = get_loss(label_smoothing=0.0)
optimizer = get_optimizer(model, optimizer='sgd', learning_rate=0.01, momentum=0.9, weight_decay=1e-4)

print(f'Loss:      {loss_fn}')
print(f'Optimizer: {optimizer}')

## 4. Train (A0 — 50 epochs)

This cell runs via `ablation_runner.py` which handles wandb init, checkpointing, and TorchScript export automatically.
You can also call `optimize()` directly below for quick interactive runs.

In [ ]:
# Option A: use the ablation runner (recommended — handles wandb + export)
!python ablation_runner.py --config configs/A0_scratch.yaml

In [ ]:
# Option B: interactive training (no wandb, no export)
# from src.train import optimize
# optimize(data_loaders, model, optimizer, loss_fn,
#          n_epochs=50, save_path='checkpoints/A0_scratch.pt')

## 5. Test evaluation

In [ ]:
import torch
from src.model import MyModel
from src.optimization import get_loss
from src.train import one_epoch_test
from sklearn.metrics import f1_score

model = MyModel(num_classes=50)
model.load_state_dict(torch.load('checkpoints/A0_scratch.pt', map_location='cpu'))

loss_fn = get_loss()
test_loss, top1, top5, targets, preds = one_epoch_test(data_loaders['test'], model, loss_fn)
macro_f1 = f1_score(targets, preds, average='macro', zero_division=0)

print(f'Top-1: {100*top1:.1f}%  |  Top-5: {100*top5:.1f}%  |  Macro-F1: {100*macro_f1:.1f}%')

## 6. Confusion matrix

In [ ]:
from src.helpers import plot_confusion_matrix

plot_confusion_matrix(preds, targets)

## 7. Export TorchScript checkpoint

In [ ]:
from src.predictor import Predictor
from src.helpers import compute_mean_and_std

class_names = data_loaders['train'].dataset.classes
mean, std   = compute_mean_and_std()

model_cpu = MyModel(num_classes=50).cpu()
model_cpu.load_state_dict(torch.load('checkpoints/A0_scratch.pt', map_location='cpu'))

predictor = Predictor(model_cpu, class_names, mean, std).cpu()
scripted  = torch.jit.script(predictor)
scripted.save('checkpoints/A0_scratch_exported.pt')
print('Saved: checkpoints/A0_scratch_exported.pt')